In [ ]:
import pandas as pd
import numpy as np
import sktime.datatypes
from loguru import logger
from data_preprocessing.utils.data_handling import load_parquet_table
from pathlib import Path
import sktime as skt 
import matplotlib.pyplot as plt


In [ ]:
notebook_path = Path.cwd()
project_root = notebook_path.parents[1]
dataset_path: Path = project_root / Path(
    "datasets/raw_and_shitty/dataset_ts.parquet")

df_dataset = load_parquet_table(parquet_path=dataset_path)

In [ ]:
df_dataset.head()

In [ ]:
from sktime.classification.interval_based import TimeSeriesForestClassifier
from sktime.datasets import load_arrow_head
from sklearn.model_selection import train_test_split
from sktime.split import temporal_train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
df_dataset_small = df_dataset.loc[:, ["patientunitstayid", "time_offset", "heartrate", "sao2", "sepsis_patient"]]

In [ ]:
df_dataset_small_without_duplicates = df_dataset_small.drop_duplicates(subset=["patientunitstayid", "time_offset"])

In [ ]:
df_dataset_small_mi_without_duplicates = df_dataset_small_without_duplicates.set_index(["patientunitstayid", "time_offset"])


Reducing the number of instances, clf.fit() will take a lot of time otherwise


In [ ]:
patient_ids = df_dataset["patientunitstayid"].unique().tolist()
df_dataset_small_mi_without_duplicates = df_dataset_small_mi_without_duplicates.loc[patient_ids[0:49], :]

In [ ]:
df_dataset_small_mi_without_duplicates.head()

In [ ]:
df_dataset_small_mi = df_dataset_small.set_index(["patientunitstayid", "time_offset"])

In [ ]:
df_dataset_small_mi.head()

In [ ]:
df_dataset_small_mi_without_duplicates.index

In [ ]:
X = df_dataset_small_mi_without_duplicates.drop(columns=["sepsis_patient"])
#y = df_dataset_small_mi_without_duplicates["sepsis_patient"]
y = df_dataset_small_mi_without_duplicates.loc[:,"sepsis_patient"]

In [ ]:
from sktime.datatypes import MTYPE_REGISTER
MTYPE_REGISTER

In [ ]:
skt.datatypes.check_is_mtype(X, "pd-multiindex")

In [ ]:
skt.datatypes.check_is_mtype(y, "pd-multiindex", return_metadata=True)

In [ ]:
skt.datatypes.check_is_mtype(y, "pd.Series", return_metadata=True)

In [ ]:
skt.datatypes.check_is_mtype(y, mtype="pd_Series_Table", return_metadata=True)

In [ ]:
y_reset = y.reset_index(drop=False)
y_new = y_reset.set_index(["patientunitstayid"])

y_only_id_sepsis = y_new.drop(columns=["time_offset"])
y_only_unique_sepsis = y_reset.drop_duplicates(subset=["patientunitstayid"])



In [ ]:
skt.datatypes.check_is_mtype(y_new, "pd-multiindex", return_metadata=True)

y must have the same number of instances as X, as i doesn't consider the time stamp where a specific classification is being made  

This seems to be the problem related to the complaints of temporal train test split, how can i make sure that y is appropriate?

Exogenous variables are features
Endogenous variables are targets

In [ ]:
# muss eventuell die duplicates noch droppen
X_train, X_test, y_train, y_test = train_test_split(X,y_only_id_sepsis)
#X_train, X_test, y_train, y_test = temporal_train_test_split(y_new, X, train_size=0.8, test_size=0.2)

In [ ]:
X_train_sorted = X_train.sort_index()

In [ ]:
from sktime.classification.kernel_based import RocketClassifier

classifier = RocketClassifier(use_multivariate="yes")

In [ ]:
classifier.fit(X_train_sorted, y_train)

In [ ]:
from sktime.registry._lookup import all_estimators
#estimator_types="classifier"
all_estimators(estimator_types="classifier")

In [ ]:
all_estimators(estimator_types="regressor")